# GPU RL Trading — EURUSD
Batched DQN training on GPU. All heavy computation is in PyTorch tensors.

In [ ]:
# ── Mount Drive & clone repo (run once) ────────────────────────────────────────────
import os, sys

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Clone or pull repo
REPO = "/content/deep-reinforcement-learning-trading"
if not os.path.exists(REPO):
    !git clone https://github.com/monty313/deep-reinforcement-learning-trading.git {REPO}
else:
    !cd {REPO} && git pull

sys.path.insert(0, REPO)

# ── Set your CSV path ───────────────────────────────────────────────────
DATA_CSV_EURUSD = "/content/drive/MyDrive/EURUSD_M1_202101131130_202605270000_2020_2026.csv"
# Adjust ^ to wherever you uploaded the CSV in Drive

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import sys
sys.path.insert(0, "/content/deep-reinforcement-learning-trading")

from gpu_rl_trading.training.train import run_training

cfg_overrides = {
    "DATA_CSV_EURUSD":  DATA_CSV_EURUSD,
    "DATE_FROM":        "2022-01-01",
    "DATE_TO":          "2023-12-31",
    "NUM_EPISODES":     20,           # quick test
    "BATCH_SIZE_ENV":   8,            # 8 parallel episodes
    "EPISODE_BARS":     21_600,       # ~15 trading days
    "BATCH_SIZE_RL":    256,
    "CHECKPOINT_EVERY": 10,
}

agent = run_training(cfg=cfg_overrides, resume=False)

In [ ]:
import pandas as pd
from pathlib import Path

ftmo_csv = Path("/content/deep-reinforcement-learning-trading/gpu_rl_trading/metrics/metrics_EURUSD_ftmo_gpu.csv")
rewards_csv = Path("/content/deep-reinforcement-learning-trading/gpu_rl_trading/metrics/episode_rewards_EURUSD_gpu.csv")

if ftmo_csv.exists():
    df_ftmo = pd.read_csv(ftmo_csv)
    print(f"\nFTMO daily metrics ({len(df_ftmo)} days logged):")
    print(df_ftmo["ftmo_flag"].value_counts().to_string())
    print(df_ftmo.tail(10).to_string(index=False))

if rewards_csv.exists():
    df_ep = pd.read_csv(rewards_csv)
    print(f"\nEpisode rewards ({len(df_ep)} episodes):")
    print(df_ep.to_string(index=False))

In [ ]:
# To resume from latest checkpoint:
agent = run_training(cfg={
    "DATA_CSV_EURUSD": DATA_CSV_EURUSD,
    "NUM_EPISODES": 10,
}, resume=True)